# RAG Architecture & Variants Notebook
Fully updated with dependency installs, consolidated imports, fixed query rewriting, RAG pipeline, and streaming example.

## 1. Setup: Install Dependencies
Install all required Python libraries including langchain-openai.

In [1]:
# Install required libraries
# pip install -r llm_requirements.txt

## 2. Imports
Import all necessary modules at the beginning of the notebook.

In [3]:
import os
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.prompts import PromptTemplate
from langchain_classic.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from ragas.metrics import (faithfulness, answer_relevancy)
import json
from collections import defaultdict
from ragas.evaluation import evaluate
from datasets import Dataset
# Updated ChatOpenAI import
from langchain_openai import ChatOpenAI

### Create a vectorstore

In [12]:
documents = [
    "In March 2023, the U.S. government capped insulin prices at $35 for seniors under Medicare Part D.",
    "Insulin price reductions were introduced following years of advocacy for affordable diabetes care.",
    "Crohn's disease is a chronic inflammatory bowel disease that affects the gastrointestinal tract.",
    "Common complications of Crohn's include strictures, fistulas, and malnutrition.",
    "In 2023, Eli Lilly announced a major price drop on several insulin products, including Humalog.",
    "RAG (Retrieval-Augmented Generation) combines language models with document retrieval for better factual grounding.",
    "Patients with Crohn's often require long-term medication and occasional surgery to manage flare-ups.",
    "The Inflation Reduction Act of 2022 contributed to broader changes in drug pricing in the United States.",
    "Conversational RAG keeps memory across user interactions for context-aware answers.",
    "Multi-hop retrieval in RAG helps answer complex queries by combining facts from multiple sources.",
    "Diagnostic criteria for Crohn's disease include clinical symptoms (abdominal pain, diarrhea, weight loss), endoscopic findings showing discontinuous inflammation with skip lesions, transmural inflammation on histology, and imaging evidence of bowel wall thickening or fistulas.",
    "Key diagnostic tests for Crohn's disease include colonoscopy with biopsy, capsule endoscopy, CT or MRI enterography, and laboratory tests such as fecal calprotectin and C-reactive protein (CRP) to assess inflammation.",
    "Crohn's disease diagnosis requires excluding other conditions like ulcerative colitis, infectious colitis, and irritable bowel syndrome through comprehensive clinical evaluation, imaging, and histopathological examination."
]
embedding_fn = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(texts=documents, embedding=embedding_fn)

In [13]:
# Verify diagnostic criteria can be retrieved
test_query = "diagnostic criteria Crohn's disease"
test_docs = vectorstore.similarity_search(test_query, k=3)
print("Retrieved documents for 'diagnostic criteria':")
for i, doc in enumerate(test_docs, 1):
    print(f"\n{i}. {doc.page_content[:150]}...")


Retrieved documents for 'diagnostic criteria':

1. Diagnostic criteria for Crohn's disease include clinical symptoms (abdominal pain, diarrhea, weight loss), endoscopic findings showing discontinuous i...

2. Crohn's disease diagnosis requires excluding other conditions like ulcerative colitis, infectious colitis, and irritable bowel syndrome through compre...

3. Key diagnostic tests for Crohn's disease include colonoscopy with biopsy, capsule endoscopy, CT or MRI enterography, and laboratory tests such as feca...


## 3. Query Rewrite Example - Setup
Define the corpus, build the FAISS vector store, and implement the query rewriting function with `invoke`.

In [14]:
def rewrite_query(original_query: str, context: str) -> str:
    """Rewrite the user query for improved retrieval, returning a plain string."""
    if not os.getenv("OPENAI_API_KEY"):
        return original_query

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)  
    prompt = PromptTemplate.from_template(
        "Rewrite the following question for better retrieval given the context.\n\n"
        "Context:\n{context}\n\nQuestion:\n{query}\n\nRewrite:"
    )
    rewritten_prompt = prompt.format(query=original_query, context=context)  
    result = llm.invoke(rewritten_prompt)
    return getattr(result, "content", str(result))

In [15]:
rewrite_query('What legislation affected insulin pricing?', "")

'What laws have impacted the pricing of insulin?'

## 4. Query Rewrite Example - RAG Pipeline
Retrieve relevant documents and generate an answer using the RAG approach with `invoke`.

In [16]:
def rag_qa(query: str, k: int = 3) -> str:
    """Retrieve top-k documents and generate an LLM-based answer (returns a string)."""
    if not isinstance(query, str) and hasattr(query, "content"):
        query = query.content
    docs = vectorstore.similarity_search(query, k=k)
    context = "\n".join(doc.page_content for doc in docs)
    print('context',context)
    if not os.getenv("OPENAI_API_KEY"):
        print("Retrieved Context:\n", context)
        return "No generation (API key not provided)"
    llm = ChatOpenAI(temperature=0.3)
    prompt = (
        f"Given the following context:\n{context}\n\n"
        f"Answer the question: {query}"
    )
    response = llm.invoke(prompt)
    return response.content if hasattr(response, "content") else str(response)

# Example usage
user_query = "What legislation"
docs = vectorstore.similarity_search(user_query, k=2)
print("docs", docs)
context = " ".join(doc.page_content for doc in docs)
# print(context)
rewritten = rewrite_query(user_query, context)
print("🔁 Rewritten Query:", rewritten)
print("\n🧠 Answer:", rag_qa(rewritten))
print("\n🧠 Answer user_query:", rag_qa(user_query))

docs [Document(id='7947d5ac-46a6-48cb-a4b6-3030ef80343b', metadata={}, page_content='The Inflation Reduction Act of 2022 contributed to broader changes in drug pricing in the United States.'), Document(id='9e20a9f5-b478-4e8d-bafe-87c78764627c', metadata={}, page_content='Insulin price reductions were introduced following years of advocacy for affordable diabetes care.')]
🔁 Rewritten Query: What legislation led to insulin price reductions and changes in drug pricing in the United States?
context Insulin price reductions were introduced following years of advocacy for affordable diabetes care.
The Inflation Reduction Act of 2022 contributed to broader changes in drug pricing in the United States.
In March 2023, the U.S. government capped insulin prices at $35 for seniors under Medicare Part D.

🧠 Answer: The Inflation Reduction Act of 2022 led to insulin price reductions and changes in drug pricing in the United States.
context The Inflation Reduction Act of 2022 contributed to broader c

## 5. Conversational RAG Example
Set up a conversational retrieval chain with memory for multi-turn interactions.

In [8]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.3),
    retriever=retriever,
    memory=memory,
)

print("User: Tell me about Crohn's disease")
print('memory', memory)
print("Assistant:", qa_chain.run("Tell me about Crohn's disease"))
print('memory 1:', memory)
print("\nUser: What are the complications?")
print("Assistant:", qa_chain.run("What are the complications?"))
print('memory 2:', memory)

User: Tell me about Crohn's disease
memory chat_memory=InMemoryChatMessageHistory(messages=[]) return_messages=True memory_key='chat_history'


C:\Users\zvibe\AppData\Local\Temp\ipykernel_47164\577550415.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
C:\Users\zvibe\AppData\Local\Temp\ipykernel_47164\577550415.py:11: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  print("Assistant:", qa_chain.run("Tell me about Crohn's disease"))


Assistant: Crohn's disease is a chronic inflammatory bowel disease that affects the gastrointestinal tract. It is characterized by symptoms such as abdominal pain, diarrhea, and weight loss. Diagnosis involves clinical symptoms, endoscopic findings, histology showing transmural inflammation, and imaging evidence of bowel wall thickening or fistulas. It is important to differentiate Crohn's disease from other conditions like ulcerative colitis, infectious colitis, and irritable bowel syndrome through a comprehensive evaluation.
memory 1: chat_memory=InMemoryChatMessageHistory(messages=[HumanMessage(content="Tell me about Crohn's disease", additional_kwargs={}, response_metadata={}), AIMessage(content="Crohn's disease is a chronic inflammatory bowel disease that affects the gastrointestinal tract. It is characterized by symptoms such as abdominal pain, diarrhea, and weight loss. Diagnosis involves clinical symptoms, endoscopic findings, histology showing transmural inflammation, and imag

In [9]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationSummaryBufferMemory

# 1) Retriever (use your existing vectorstore)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 2) LLMs: one for answers, one deterministic for summaries
gen_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
sum_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3) Token-aware memory (auto-summarizes older turns when over budget)
memory = ConversationSummaryBufferMemory(
    llm=sum_llm,
    memory_key="chat_history",
    return_messages=True,
    max_token_limit=200,   # small to force summarization for the demo
)

# 4) Conversational Retrieval Chain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=gen_llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=False,
)

def ask(q: str):
    print(f"\nUser: {q}")
    resp = qa_chain.invoke({"question": q})
    print("Assistant:", resp["answer"])
    # Show when summarization kicks in
    summary = getattr(memory, "moving_summary_buffer", "")
    if summary:
        print("\n[Memory summary present]")
        print(summary[:220] + ("..." if len(summary) > 220 else ""))

# ---- Demo turns (long prompts to exceed the tiny token budget) ----
ask("Tell me about Crohn's disease in detail (at least 150 words).")
ask("What are the complications? Provide a comprehensive overview (150+ words).")
ask("Briefly summarize the key diagnostic criteria.")



User: Tell me about Crohn's disease in detail (at least 150 words).


C:\Users\zvibe\AppData\Local\Temp\ipykernel_47164\2011397976.py:13: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationSummaryBufferMemory(


Assistant: Crohn's disease is a chronic inflammatory bowel disease (IBD) that primarily affects the gastrointestinal tract, leading to a range of symptoms and complications. It can occur anywhere along the digestive tract, from the mouth to the anus, but is most commonly found in the ileum (the last part of the small intestine) and the colon. The inflammation in Crohn's disease is often discontinuous, characterized by "skip lesions," where healthy tissue is interspersed with inflamed areas.

Patients with Crohn's disease typically experience symptoms such as abdominal pain, diarrhea (which may be bloody), weight loss, and fatigue. These symptoms can significantly impact quality of life and may lead to complications like bowel obstructions, fistulas, and malnutrition.

Diagnosis involves a comprehensive evaluation, including clinical assessment, endoscopic procedures that reveal the characteristic inflammation, imaging studies that show bowel wall thickening, and histological examinatio

### 5.1 Short and long term memory

#### Save a fact into memory

In [10]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory, VectorStoreRetrieverMemory
from langchain_classic.prompts import PromptTemplate
from langchain_community.vectorstores import FAISS
import os
import re

# --- Persistence setup ---
os.makedirs("local_data", exist_ok=True)
FAISS_DIR = "local_data/memory_store"

embeddings = OpenAIEmbeddings()

# Load if exists, else create new
if os.path.isdir(FAISS_DIR):
    vectorstore = FAISS.load_local(FAISS_DIR, embeddings, allow_dangerous_deserialization=True)
else:
    vectorstore = FAISS.from_texts(["(memory anchor)"], embeddings)
    vectorstore.save_local(FAISS_DIR)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
gen_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

# Short-term memory for CRC (message format)
chat_memory = ConversationBufferMemory(
    memory_key="chat_history",
    input_key="question",
    output_key="answer",
    return_messages=True,
)

# Long-term memory (string format)
longterm_memory = VectorStoreRetrieverMemory(
    retriever=retriever,
    memory_key="history",
    input_key="question",
    output_key="answer",
)

# Custom prompt that instructs the LLM to remember information
custom_prompt = PromptTemplate(
    input_variables=["chat_history", "question", "context"],
    template="""You are a helpful assistant with memory capabilities. This is a memory-enabled system where you should acknowledge and remember user preferences and information when asked.

IMPORTANT: When users ask you to remember something, you should acknowledge it positively (e.g., "I'll remember that your favorite color is blue" or "Got it, I've noted that"). This system will automatically save your responses to long-term memory.

Use the following pieces of context to answer the question. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}

Chat History:
{chat_history}

Question: {question}

Helpful Answer:"""
)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=gen_llm,
    retriever=retriever,
    memory=chat_memory,
    return_source_documents=False,
    combine_docs_chain_kwargs={"prompt": custom_prompt},
)

def persist():
    vectorstore.save_local(FAISS_DIR)

def extract_remember_info(question: str) -> str:
    """Extract information from 'remember' requests for direct storage."""
    # Pattern to match "Remember that X" or "X. Remember that."
    patterns = [
        r"(.+?)\s*[.,]\s*[Rr]emember\s+that[.]?",
        r"[Rr]emember\s+that\s+(.+?)[.]?",
        r"[Rr]emember:\s*(.+?)[.]?",
    ]
    for pattern in patterns:
        match = re.search(pattern, question, re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return None

def ask(q: str):
    # Retrieve long-term memory and inject into the question
    mem_txt = longterm_memory.load_memory_variables({"question": q}).get("history", "")
    augmented_q = f"[MEMORY]\n{mem_txt}\n\n[QUESTION]\n{q}" if mem_txt.strip() else q

    print(f"\nUser: {q}")
    resp = qa_chain.invoke({"question": augmented_q})
    answer = resp["answer"]
    print("Assistant:", answer)

    # If the LLM refused to remember, extract info directly from the question
    if "can't remember" in answer.lower() or "don't remember" in answer.lower():
        remembered_info = extract_remember_info(q)
        if remembered_info:
            # Create a positive acknowledgment response
            answer = f"I'll remember that {remembered_info.lower()}."
            print(f"Assistant (corrected): {answer}")

    # Save this new turn into long-term memory + persist FAISS
    longterm_memory.save_context({"question": q}, {"answer": answer})
    persist()

# ---- Demo ----
ask("I was in New York last year. Remember that.")


C:\Users\zvibe\AppData\Local\Temp\ipykernel_47164\2390767638.py:34: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  longterm_memory = VectorStoreRetrieverMemory(



User: I was in New York last year. Remember that.
Assistant: Got it, I've noted that you were in New York last year.


#### Reload and Retrieve

In [11]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory, VectorStoreRetrieverMemory
from langchain_community.vectorstores import FAISS

FAISS_DIR = "local_data/memory_store"
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.load_local(FAISS_DIR, embeddings, allow_dangerous_deserialization=True)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
gen_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

chat_memory = ConversationBufferMemory(
    memory_key="chat_history",
    input_key="question",
    output_key="answer",
    return_messages=True,
)

longterm_memory = VectorStoreRetrieverMemory(
    retriever=retriever,
    memory_key="history",
    input_key="question",
    output_key="answer",
)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=gen_llm,
    retriever=retriever,
    memory=chat_memory,
    return_source_documents=False,
)

def ask(q: str, use_mem_prefix: bool = True):
    mem_txt = longterm_memory.load_memory_variables({"question": q}).get(longterm_memory.memory_key, "")
    augmented_q = f"[MEMORY]\n{mem_txt}\n\n[QUESTION]\n{q}" if (use_mem_prefix and mem_txt.strip()) else q

    print(f"\nUser: {q}")
    resp = qa_chain.invoke({"question": augmented_q})
    print("Assistant:", resp["answer"])

# ✅ Should recall “teal” thanks to saved FAISS memory
ask("where have I been last year?")



User: where have I been last year?
Assistant: You were in New York last year.


## 6. ReAct + RAG Example
Demonstrate a ReAct agent using a retriever tool for hybrid reasoning.

In [21]:
documents = [
    "In March 2023, the U.S. government capped insulin prices at $35 for seniors under Medicare Part D.",
    "Insulin price reductions were introduced following years of advocacy for affordable diabetes care.",
    "Crohn's disease is a chronic inflammatory bowel disease that affects the gastrointestinal tract.",
    "Common complications of Crohn's include strictures, fistulas, and malnutrition.",
    "In 2023, Eli Lilly announced a major price drop on several insulin products, including Humalog.",
    "RAG (Retrieval-Augmented Generation) combines language models with document retrieval for better factual grounding.",
    "Patients with Crohn's often require long-term medication and occasional surgery to manage flare-ups.",
    "The Inflation Reduction Act of 2022 contributed to broader changes in drug pricing in the United States.",
    "Conversational RAG keeps memory across user interactions for context-aware answers.",
    "Multi-hop retrieval in RAG helps answer complex queries by combining facts from multiple sources.",
    "Diagnostic criteria for Crohn's disease include clinical symptoms (abdominal pain, diarrhea, weight loss), endoscopic findings showing discontinuous inflammation with skip lesions, transmural inflammation on histology, and imaging evidence of bowel wall thickening or fistulas.",
    "Key diagnostic tests for Crohn's disease include colonoscopy with biopsy, capsule endoscopy, CT or MRI enterography, and laboratory tests such as fecal calprotectin and C-reactive protein (CRP) to assess inflammation.",
    "Crohn's disease diagnosis requires excluding other conditions like ulcerative colitis, infectious colitis, and irritable bowel syndrome through comprehensive clinical evaluation, imaging, and histopathological examination."
]
embedding_fn = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(texts=documents, embedding=embedding_fn)

In [23]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

# 1. Tool: Wrap your vectorstore retriever
@tool
def insight_search(query: str) -> str:
    """
    Searches the document base for medical information.
    Answer ONLY using the retrieved documents. 
    If no relevant documents found, return 'NO_RESULTS'.
    """
    retriever = vectorstore.as_retriever()
    docs = retriever.invoke(query)
    if not docs:
        return "NO_RESULTS"
    return "\n\n".join(d.page_content for d in docs)

# 2. LLM
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0)

# 3. Create the agent 
agent = create_agent(
    model=llm,
    tools=[insight_search],
    system_prompt=(
        "You are a medical assistant.\n"
        "- Use the tool `insight_search` whenever needed.\n"
        "- Answer ONLY based on retrieved documents.\n"
        "- If the documents do not contain the answer, say: "
        "'I don't know based on the available documents.'"
    ),
)

# 4. Ask your question
query = (
    "What are the new insulin pricing changes announced in 2023"
)
# query = (
#     "What are do you know about Golden Retriver dogs?"
#     )

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

response = agent.invoke({
    "messages": [{"role": "user", "content": query}]
})

# 5. Print final answer (last assistant message)
print(response["messages"][-1].content)


================================ Human Message =================================

What are the new insulin pricing changes announced in 2023
================================== Ai Message ==================================
Tool Calls:
  insight_search (call_fChfTEhduEHGI7x3OELhoYt9)
 Call ID: call_fChfTEhduEHGI7x3OELhoYt9
  Args:
    query: new insulin pricing changes announced in 2023
================================= Tool Message =================================
Name: insight_search

In March 2023, the U.S. government capped insulin prices at $35 for seniors under Medicare Part D.

Insulin price reductions were introduced following years of advocacy for affordable diabetes care.

In 2023, Eli Lilly announced a major price drop on several insulin products, including Humalog.

The Inflation Reduction Act of 2022 contributed to broader changes in drug pricing in the United States.
================================== Ai Message ==================================

In 2023, the U.S. governm

## 7. Multi-hop Retrieval Example
Illustrate sequential retrieval hops for complex queries.

In [6]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def search_docs(query: str) -> str:
    """Search the medical/legislation document base. Call multiple times to chain facts."""
    docs = vectorstore.as_retriever(search_kwargs={"k": 2}).invoke(query)
    return "\n\n".join(d.page_content for d in docs) if docs else "NO_RESULTS"

agent = create_agent(
    model=ChatOpenAI(model="gpt-3.5-turbo", temperature=0.0),
    tools=[search_docs],
    system_prompt=(
        "Answer using the document base. Questions may require MULTIPLE searches: "
        "search once, read the result, then search again for follow-up facts it reveals "
        "before answering."
    ),
)

resp = agent.invoke({"messages": [{"role": "user",
    "content": "How did U.S. legislation affect insulin pricing? Trace the specific law and its provisions."}]})

# Show the hops (the reasoning trace from our earlier discussion)
for m in resp["messages"]:
    m.pretty_print()


================================ Human Message =================================

How did U.S. legislation affect insulin pricing? Trace the specific law and its provisions.
================================== Ai Message ==================================
Tool Calls:
  search_docs (call_mxdtug8log63y7nZtFVKfZRz)
 Call ID: call_mxdtug8log63y7nZtFVKfZRz
  Args:
    query: insulin pricing U.S. legislation
================================= Tool Message =================================
Name: search_docs

In March 2023, the U.S. government capped insulin prices at $35 for seniors under Medicare Part D.

Insulin price reductions were introduced following years of advocacy for affordable diabetes care.
================================== Ai Message ==================================
Tool Calls:
  search_docs (call_qE8gq0CdGvm7KUD2NBqYspDt)
 Call ID: call_qE8gq0CdGvm7KUD2NBqYspDt
  Args:
    query: Medicare Part D insulin pricing legislation
================================= Tool Message =======

## 8. Multi-vector Retrieval Example
Embed multiple parts of documents separately for finer-grained search.

In [8]:
import uuid
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

# 1. PARENT docs — the full text we want the LLM to see
parents = [
    {"title": "Insulin Price Cap",
     "body":  "In March 2023, the U.S. government capped insulin prices at $35 for Medicare seniors."},
    {"title": "Crohn's Complications",
     "body":  "Common complications of Crohn's include strictures, fistulas, and malnutrition."},
    {"title": "Eli Lilly Insulin",
     "body":  "Eli Lilly reduced prices of insulin in 2023, including Humalog."},
]

# 2. docstore = a plain dict mapping id -> full parent body
docstore = {}
child_docs = []
for p in parents:
    pid = str(uuid.uuid4())
    docstore[pid] = p["body"]                          # <-- the parent lookup table

    # MULTIPLE child vectors per parent, each tagged with the parent id
    for rep in [p["title"], p["body"], f"Question this answers: {p['title']}?"]:
        child_docs.append(Document(page_content=rep, metadata={"doc_id": pid}))

# 3. FAISS holds only the small child vectors (you already have this)
vectorstore = FAISS.from_documents(child_docs, embedding=embedding_fn)


In [9]:
def multi_vector_search(query, k=4):
    """Search child vectors, then return the unique PARENT docs they point to."""
    child_hits = vectorstore.similarity_search(query, k=k)
    seen, parents_out = set(), []
    for child in child_hits:
        pid = child.metadata["doc_id"]
        if pid not in seen:                # dedupe: many children -> one parent
            seen.add(pid)
            parents_out.append(docstore[pid])
    return parents_out, child_hits


In [10]:
query = "Which company lowered insulin prices and why?"
parents_out, child_hits = multi_vector_search(query)

print("WHAT MATCHED (child vectors):")
for c in child_hits:
    print("  -", c.page_content)

print("\nWHAT THE LLM GETS (parent docs):")
for body in parents_out:
    print("  -", body)


WHAT MATCHED (child vectors):
  - Eli Lilly reduced prices of insulin in 2023, including Humalog.
  - Question this answers: Insulin Price Cap?
  - Insulin Price Cap
  - In March 2023, the U.S. government capped insulin prices at $35 for Medicare seniors.

WHAT THE LLM GETS (parent docs):
  - Eli Lilly reduced prices of insulin in 2023, including Humalog.
  - In March 2023, the U.S. government capped insulin prices at $35 for Medicare seniors.


In [11]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.3)
context = "\n".join(parents_out)
answer = llm.invoke(f"Context:\n{context}\n\nQuestion: {query}\nAnswer:").content
print("\nAnswer:", answer)



Answer: Eli Lilly lowered insulin prices in 2023, including Humalog, in response to pressure to make insulin more affordable for patients.


## 9. Streaming RAG Example
Configure a streaming-capable LLM and include context for Streaming RAG.

In [24]:
# Add a document defining Streaming RAG
documents.append(
    "Streaming RAG is a variant of Retrieval-Augmented Generation where the language model emits tokens as they are generated, while retrieval provides relevant context in real time. This reduces perceived latency and improves user engagement."
)
# Rebuild vectorstore
vectorstore = FAISS.from_texts(texts=documents, embedding=embedding_fn)
retriever = vectorstore.as_retriever()

# Configure streaming LLM
streaming_llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    temperature=0.2,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)
qa_chain = RetrievalQA.from_chain_type(llm=streaming_llm, retriever=retriever)
print("Assistant:", end=" ")
qa_chain.run("What is Streaming RAG and why is it useful?")

Assistant: Streaming RAG is a variant of Retrieval-Augmented Generation where the language model emits tokens as they are generated, while retrieval provides relevant context in real time. This reduces perceived latency and improves user engagement. It is useful because it allows for a more seamless and interactive experience by providing relevant context as the language model generates responses, enhancing user engagement and reducing perceived delays.

'Streaming RAG is a variant of Retrieval-Augmented Generation where the language model emits tokens as they are generated, while retrieval provides relevant context in real time. This reduces perceived latency and improves user engagement. It is useful because it allows for a more seamless and interactive experience by providing relevant context as the language model generates responses, enhancing user engagement and reducing perceived delays.'

## 10. RAGAS Evaluation Example
Evaluate RAG performance using RAGAS metrics on a test dataset.

In [17]:
samples = [
    # 1) Fully supported, specific answer (should score high on everything)
    {
        "question": "What is Crohn's disease?",
        "answer": (
            "Crohn's disease is a chronic inflammatory bowel disease that can affect any part "
            "of the GI tract from mouth to anus, with transmural inflammation and skip lesions."
        ),
        "ground_truths": [
            "Crohn's disease is a chronic inflammatory bowel disease that can affect any part of the gastrointestinal tract; it is transmural and has skip lesions."
        ],
        "contexts": [
            "Crohn's disease may involve any segment of the GI tract and shows transmural inflammation.",
            "Skip lesions are typical of Crohn's disease."
        ],
    },

    # 2) Hallucination / incorrect therapy (should tank faithfulness & correctness)
    {
        "question": "What is fist-line induction therapy for moderate–severe Crohn's?",
        "answer": "Aspirin and broad-spectrum antibiotics are first-line for induction.",
        "ground_truths": [
            "Induction therapy for moderate–severe Crohn's commonly uses systemic corticosteroids or biologics such as anti-TNF (e.g., infliximab); aspirin is not indicated."
        ],
        "contexts": [
            "For moderate–severe Crohn’s disease, induction options include systemic corticosteroids or biologic therapy such as anti-TNF (infliximab).",
            "Aspirin is not a therapy for Crohn's disease."
        ],
    },

    # 3) Too vague / generic answer (low answer_relevancy; context is detailed)
    {
        "question": "How is fecal calprotectin used in IBD?",
        "answer": "It's a stool test doctors sometimes order.",
        "ground_truths": [
            "Fecal calprotectin helps distinguish IBD from IBS and monitor intestinal inflammation; values >250 μg/g suggest active IBD, <50 μg/g makes active inflammation unlikely."
        ],
        "contexts": [
            "Fecal calprotectin correlates with mucosal inflammation; values >250 μg/g suggest active disease.",
            "Values <50 μg/g make active IBD unlikely and point toward functional disorders such as IBS."
        ],
    },

    # 4) Correct fact with mild distractor context (tests precision/recall)
    {
        "question": "Which complication is more common in Crohn's disease than in ulcerative colitis?",
        "answer": "Fistulas.",
        "ground_truths": [
            "Fistulas are more common in Crohn's disease than in ulcerative colitis."
        ],
        "contexts": [
            "Penetrating complications such as fistulas and strictures are characteristic of Crohn's disease.",
            "Ulcerative colitis is limited to the colon and confined to the mucosa."
        ],
    },

    # 5) Direct contradiction to context (tests faithfulness & correctness)
    {
        "question": "Which part of the GI tract is affected in ulcerative colitis?",
        "answer": "It causes skip lesions that can affect any part of the GI tract.",
        "ground_truths": [
            "Ulcerative colitis involves continuous mucosal inflammation starting in the rectum and extending proximally through the colon; it is limited to the colon."
        ],
        "contexts": [
            "Ulcerative colitis is limited to the colon and involves continuous mucosal inflammation starting at the rectum.",
            "Crohn's disease may involve any GI segment and has skip lesions."
        ],
    },
     # 6) Correct answer but NOT supported by provided contexts (faithfulness ↓, correctness can be OK)
    {
        "question": "Which biologic targets IL-12/23 for Crohn's induction?",
        "answer": "Ustekinumab.",
        "ground_truths": [
            "Ustekinumab targets IL-12/23 and is used for induction and maintenance in Crohn's disease."
        ],
        "contexts": [
            # Intentionally relevant-to-IBD but not supporting the specific claim
            "Anti-TNF agents like infliximab are commonly used for Crohn’s disease.",
            "Ulcerative colitis is limited to the colon with continuous mucosal inflammation."
        ],
    },

    # 7) Multiple acceptable gold phrasings (answer should match either)
    {
        "question": "Where does ulcerative colitis typically begin?",
        "answer": "It starts in the rectum and extends proximally in a continuous pattern.",
        "ground_truths": [
            "Ulcerative colitis begins in the rectum and extends proximally in a continuous manner.",
            "UC starts at the rectum with continuous colonic involvement."
        ],
        "contexts": [
            "Ulcerative colitis is limited to the colon and typically begins in the rectum with continuous spread proximally."
        ],
    },

    # 8) Retrieval with lots of distractors (precision should suffer if model over-cites)
    {
        "question": "Name one penetrating complication characteristic of Crohn’s disease.",
        "answer": "Fistulas.",
        "ground_truths": [
            "Penetrating complications such as fistulas are characteristic of Crohn’s disease."
        ],
        "contexts": [
            "Irritable bowel syndrome is a functional disorder without mucosal inflammation.",
            "The pancreas secretes digestive enzymes into the duodenum.",
            "Fistulas and strictures are penetrating complications seen in Crohn’s disease.",  # only this is relevant
            "Peptic ulcers are typically related to H. pylori or NSAIDs.",
            "The liver receives blood from the portal vein.",
        ],
    },
]

dataset = Dataset.from_list(samples)

# Create LLM and embeddings for RAGAS evaluation
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

results = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=llm,
    embeddings=embeddings,
)

print("📊 RAGAS Metrics:", results)

Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]

📊 RAGAS Metrics: {'faithfulness': 0.4688, 'answer_relevancy': 0.4603}


### Evaluate using LLM as a Judge

In [18]:
# pip install langchain langchain-openai pydantic
from typing import List, Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# ---------- Structured schema ----------
class GroundingJudgment(BaseModel):
    grounded_score: int = Field(ge=1, le=5, description="5 = fully grounded in context")
    grounded: bool
    factual_score: int = Field(ge=1, le=5, description="5 = factually correct")
    factual: bool
    evidence_quotes: List[str] = Field(description="Direct quotes from CONTEXT supporting the answer")
    missing_evidence_notes: str = Field(description="What evidence is missing in CONTEXT, if any")
    reasoning: str = Field(description="Brief rationale for both scores")

# ---------- Prompt ----------
rubric = """
You are a strict evaluator of a QA system. Judge ONLY the provided data.

Scoring rules:
- Groundedness (1-5): Is the answer fully supported by the CONTEXT?
  5 = fully supported with explicit matching evidence
  3 = partially supported / requires inference beyond context
  1 = unsupported or contradicted by context
- Factuality (1-5): Is the answer factually correct given the CONTEXT and widely-accepted knowledge?
  5 = correct; 3 = partly correct or incomplete; 1 = incorrect

Return JSON ONLY matching the schema.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", rubric),
    ("human",
     "CONTEXT:\n{context}\n\nQUESTION:\n{question}\n\nANSWER:\n{answer}\n\n"
     "Produce a strict JSON object adhering to the provided schema.")
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # pick your model

# Bind the output to the Pydantic schema:
judge_chain = prompt | llm.with_structured_output(GroundingJudgment)


In [19]:
# ---------- Example usage ----------
context = (
    "Infliximab is an anti-TNF biologic used for moderate-to-severe Crohn's disease. "
    "First-line therapy may include corticosteroids for induction; maintenance can use biologics."
)
question = "What is a first-line therapy for moderate-to-severe Crohn's disease?"
answer   = "Infliximab is commonly used; steroids can induce remission, biologics maintain it."

result: GroundingJudgment = judge_chain.invoke(
    {"context": context, "question": question, "answer": answer}
)
print('Judge evaluation:', result)

Judge evaluation: grounded_score=5 grounded=True factual_score=5 factual=True evidence_quotes=["Infliximab is an anti-TNF biologic used for moderate-to-severe Crohn's disease.", 'First-line therapy may include corticosteroids for induction; maintenance can use biologics.'] missing_evidence_notes='' reasoning="The answer is fully supported by the context, which explicitly states that corticosteroids can be used for induction in first-line therapy and that infliximab is an anti-TNF biologic used for Crohn's disease. The answer is also factually correct."


In [20]:
context = (
    "Infliximab is an anti-TNF biologic used for moderate-to-severe Crohn's disease. "
    "First-line therapy may include corticosteroids for induction; maintenance can use biologics. "
    "Antibiotics are not considered first-line for induction of remission in moderate-to-severe disease."
)

question = "What is a first-line therapy for moderate-to-severe Crohn's disease?"

# ❌ Deliberately poor/incorrect answer relative to the context
answer = (
    "Antibiotics such as metronidazole are the primary first-line therapy, and surgery usually cures Crohn's. "
    "Biologics are optional and rarely needed."
)

result: GroundingJudgment = judge_chain.invoke(
    {"context": context, "question": question, "answer": answer}
)

print(result.model_dump_json(indent=2))

{
  "grounded_score": 1,
  "grounded": false,
  "factual_score": 1,
  "factual": false,
  "evidence_quotes": [
    "\"First-line therapy may include corticosteroids for induction; maintenance can use biologics.\"",
    "\"Antibiotics are not considered first-line for induction of remission in moderate-to-severe disease.\""
  ],
  "missing_evidence_notes": "The context does not support the claim that antibiotics are a first-line therapy; it explicitly states they are not considered first-line.",
  "reasoning": "The answer contradicts the context, which specifies that corticosteroids are the first-line therapy and that antibiotics are not considered first-line. Therefore, both groundedness and factuality scores are low."
}


In [21]:

# 1) Generate an answer using ONLY the provided context
gen_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer USING ONLY the provided CONTEXT. If absent, say: 'I don't know from the provided context.'"),
    ("human",  "CONTEXT:\n{context}\n\nQUESTION:\n{question}")
])
answer_chain = answer_prompt | gen_llm

context = (
    "Infliximab is an anti-TNF biologic used for moderate-to-severe Crohn's disease. "
    "First-line therapy may include corticosteroids for induction; maintenance can use biologics."
)
question = "What is a first-line therapy for moderate-to-severe Crohn's disease?"

answer_msg = answer_chain.invoke({"context": context, "question": question})
answer = getattr(answer_msg, "content", str(answer_msg))

# 2) Evaluate groundedness & factuality
judgment = judge_chain.invoke({"context": context, "question": question, "answer": answer})
print("Answer:", answer)
print(judgment.model_dump_json(indent=2))


Answer: A first-line therapy for moderate-to-severe Crohn's disease may include corticosteroids for induction.
{
  "grounded_score": 5,
  "grounded": true,
  "factual_score": 5,
  "factual": true,
  "evidence_quotes": [
    "First-line therapy may include corticosteroids for induction."
  ],
  "missing_evidence_notes": "",
  "reasoning": "The answer is fully supported by the context, which explicitly states that corticosteroids are included in first-line therapy for moderate-to-severe Crohn's disease. Additionally, the information is factually correct as corticosteroids are commonly used for induction in such cases."
}


### Use a Different Model as a Judge

In [22]:
#%pip install langchain langchain-openai langchain-google-genai google-generativeai

In [23]:
# --- Setup ---
import os, json
from typing import List
from pydantic import BaseModel, Field, ValidationError
import re
import google.generativeai as genai
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


genai.configure(api_key=os.environ["GEMINI_API_KEY"])
gemini = genai.GenerativeModel("gemini-2.0-flash-lite")

# ---------- Judge schema (GPT) ----------
class GroundingJudgment(BaseModel):
    grounded_score: int = Field(ge=1, le=5)
    grounded: bool
    factual_score: int = Field(ge=1, le=5)
    factual: bool
    evidence_quotes: List[str]
    missing_evidence_notes: str
    reasoning: str

rubric = (
    "You are a strict evaluator of a QA system. Judge ONLY the provided data.\n"
    "Groundedness (1-5): 5=fully supported by CONTEXT; 3=partial; 1=unsupported/contradicted.\n"
    "Factuality (1-5): 5=correct; 3=partly correct/incomplete; 1=incorrect.\n"
    "Return JSON ONLY with keys: grounded_score, grounded, factual_score, factual, "
    "evidence_quotes, missing_evidence_notes, reasoning. Booleans MUST be true/false."
)

judge_prompt = ChatPromptTemplate.from_messages([
    ("system", rubric),
    ("human",
     "CONTEXT:\n{context}\n\nQUESTION:\n{question}\n\nANSWER:\n{answer}\n\n"
     "Produce a strict JSON object adhering to the schema.")
])

# GPT judge with structured output → returns a Pydantic object directly
gpt_judge = ChatOpenAI(model="gpt-4o-mini", temperature=0.0, timeout=20, max_retries=1)
judge_chain = judge_prompt | gpt_judge.with_structured_output(GroundingJudgment)

# ---------- Astronomy context & question ----------
context = (
    "In exoplanet detection, the radial velocity (Doppler) method measures periodic shifts "
    "in a star's spectral lines caused by the star's line-of-sight velocity wobble due to an orbiting planet. "
    "The transit method measures the dimming of starlight when a planet crosses the stellar disk; "
    "the transit depth is approximately the area ratio (Rp/Rs)^2. "
    "The Sun is a G2V star and is cooler than F-type stars."
)
question = "What does the radial velocity method measure in exoplanet detection?"

# 1) Generate with Gemini 2.0 Flash Lite (using ONLY context)
gen_prompt = (
    "Answer USING ONLY the provided CONTEXT. If absent, say: 'I don't know from the provided context.'\n\n"
    f"CONTEXT:\n{context}\n\nQUESTION:\n{question}"
)
gen_resp = gemini.generate_content(
    gen_prompt,
    generation_config={"temperature": 0.2, "max_output_tokens": 200}
)

answer = (getattr(gen_resp, "text", None) or "").strip()
if not answer:
    # Optional: inspect for safety blocks
    details = {
        "prompt_feedback": getattr(gen_resp, "prompt_feedback", None),
        "candidates_len": len(getattr(gen_resp, "candidates", []) or [])
    }
    raise RuntimeError(f"Gemini returned empty text. Details: {json.dumps(details, default=str)}")
print("Answer:", answer)

# 2) Judge with GPT (structured JSON → Pydantic object)
judgment = judge_chain.invoke({"context": context, "question": question, "answer": answer})
print(judgment.model_dump_json(indent=2))

Answer: The radial velocity (Doppler) method measures periodic shifts in a star's spectral lines caused by the star's line-of-sight velocity wobble due to an orbiting planet.
{
  "grounded_score": 5,
  "grounded": true,
  "factual_score": 5,
  "factual": true,
  "evidence_quotes": [
    "The radial velocity (Doppler) method measures periodic shifts in a star's spectral lines caused by the star's line-of-sight velocity wobble due to an orbiting planet."
  ],
  "missing_evidence_notes": "",
  "reasoning": "The answer directly restates the information provided in the context, accurately reflecting the definition and function of the radial velocity method in exoplanet detection."
}
